## Vector stores and retrievers

# Vector stores and retrievers

Here the abstractions used in LangChain for retrievers and vector stores. The primary purpose of these tools is to enable data fetching from various origins, such as vector databases, allowing smooth integration into LLM pipelines. This functionality is essential for programs that need to retrieve context for the model to process during inference, such as in retrieval-augmented generation (RAG) setups.

We will cover:

* Documents
* Vector stores
* Retrievers

### Documents
LangChain implements a Document abstraction, which is intended to represent a unit of text and
associated metadata. It has two attributes:

- page_content: a string representing the content;
- metadata: a dict containing arbitrary metadata.
The metadata attribute can capture information about the source of the document, its relationship
to other documents, and other information. Note that an individual Document object often
represents a chunk of a larger document.

Let's generate some sample documents:

In [6]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
    page_content="Rabbits are social animals that need plenty of space to hop around.",
    metadata={"source": "mammal-pets-doc"},
    ),
]

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")
from langchain_groq import ChatGroq
llm=ChatGroq(model="openai/gpt-oss-20b",groq_api_key=groq_api_key)
llm

from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/opt/homebrew/Caskroom/miniconda/base/envs/langchain_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9074.00it/s]


In [7]:
from langchain_chroma import Chroma
vectorstore=Chroma.from_documents(documents,embedding=embeddings)
vectorstore

In [9]:
vectorstore.similarity_search("cat")


[Document(id='670614c4-03df-400e-aead-574ddd22cec1', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='1619348e-50a3-4d94-bf2a-8154970fda4e', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='36ec21dd-4359-457b-8849-68b81d866876', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='c46b33bb-b46b-4e99-9404-0ce869ddddb1', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

## Async Query

In [10]:
await vectorstore.asimilarity_search("cat")

[Document(id='670614c4-03df-400e-aead-574ddd22cec1', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='1619348e-50a3-4d94-bf2a-8154970fda4e', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='36ec21dd-4359-457b-8849-68b81d866876', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='c46b33bb-b46b-4e99-9404-0ce869ddddb1', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [11]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever=RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["cat","dog"])

[[Document(id='670614c4-03df-400e-aead-574ddd22cec1', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='1619348e-50a3-4d94-bf2a-8154970fda4e', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

Vectorstores implement an as_retriever method that will generate a Retriever, specifically a
VectorStoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify
what methods of the underlying vector store to call, and how to parameterize them. For instance, we can
replicate the above with the following:

In [12]:
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)
retriever.batch(["cat","dog"])

[[Document(id='670614c4-03df-400e-aead-574ddd22cec1', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='1619348e-50a3-4d94-bf2a-8154970fda4e', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [15]:
## RAG

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""

prompt= ChatPromptTemplate.from_messages([("human",message)])

rag_chain={"context":retriever,"question":RunnablePassthrough()}|prompt|llm

response = rag_chain.invoke("tell me about dogs.")
print(response)

content='Dogs are great companions, known for their loyalty and friendliness.' additional_kwargs={'reasoning_content': 'We need to answer using the provided context only. The context contains one document: "Dogs are great companions, known for their loyalty and friendliness." So answer: dogs are great companions, loyal, friendly. Use that.'} response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 145, 'total_tokens': 211, 'completion_time': 0.073641629, 'completion_tokens_details': {'reasoning_tokens': 45}, 'prompt_time': 0.007311038, 'prompt_tokens_details': None, 'queue_time': 0.363483481, 'total_time': 0.080952667}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_feb9b278f1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a086cc-dd8a-7ad1-b2ae-0f64da67a871-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 145, 'output_tokens': 66, 'total_tokens': 211, 'output_toke